# **Track Network - Bogie Drop Pit Lubrication**

### Data Fetching

In [2]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        # Connect to PostgreSQL
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        # Create query
        query = f"SELECT * FROM {table_name};"

        # Load into pandas DataFrame
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.95.110.69


C:\Users\win 11\AppData\Local\Temp\ipykernel_19844\3393819530.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 7760 rows from 'extraction'


In [3]:
keywords = ["BogieDropPit1Lub", "BogieDropPit2Lub"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Track-Network') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df.head(5)


,filename,workorder_id,json_data
428,TN_PM_MTH_BogieDropPit1Lub_4000495745.pdf,4.000496e+09,"{'notification': {'notification_no': 'NA', 'no..."
977,TN_PM_MTH_BogieDropPit1Lub_4000693797.pdf,4.000694e+09,"{'notification': {'notification_no': 'NA', 'no..."
1752,TN_PM_MTH_BogieDropPit1Lub_4000479302.pdf,4.000479e+09,"{'notification': {'notification_no': 'NA', 'no..."
1865,TN_PM_MTH_BogieDropPit1Lub_4000459556.pdf,4.000460e+09,"{'notification': {'notification_no': 'NA', 'no..."
2250,TN_PM_MTH_BogieDropPit2Lub_4000473467.pdf,4.000473e+09,"{'notification': {'notification_no': 'NA', 'no..."


In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_like(val):
    """Detect NA-like values."""
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    """Return keys in dict where value is NA-like."""
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]


def clean_value(val):
    """Recursively clean NA-like values in dict, list, string."""
    if isinstance(val, str):
        return '' if pattern_na.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

def extract_leaf_keys(d, parent=''):
    """Extract flattened leaf keys from nested dict."""
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                keys.extend(extract_leaf_keys(v, full_key))
            else:
                keys.append(full_key)
    return keys

### Bogie Drop Pit Lubrication

In [20]:
import string, re

def clean_component(name):
    name = name.lower().strip()
    name = re.sub(r"[^\w]+", "_", name)
    return name

def flatten_json(data):
    flat = {}

    bogie = data.get("bogie_drop_pit_lubrication")
    if isinstance(bogie, dict):
        for _, obj in bogie.items():
            if isinstance(obj, dict):
                component = clean_component(obj.get("component"))
                if not component:
                    continue

                for k, v in obj.items():
                    if k == "component":
                        continue

                    clean_k = "completed" if k == "completed?" else k
                    flat[f"{component}.{clean_k}"] = v

    technician = data.get("technician")
    if isinstance(technician, dict):
        flat["technician.technician_id"] = technician.get("technicians")
        flat["technician.date"] = technician.get("date")

    supervisor = data.get("guideway")
    if isinstance(supervisor, dict):
        flat["supervisor.supervisor_id"] = supervisor.get("supervisor")
        flat["supervisor.date"] = supervisor.get("date")

    flat["any_additional_work_that_requires_planning"] = (
        data.get("any_additional_work_that_requires_planning")
    )

    return flat

df_bogie = df.copy()

df_bogie['bogie_drop_pit_lubrication'] = df_bogie['json_data'].apply(
    lambda x: x.get('bogie_drop_pit_lubrication') if isinstance(x, dict) else None
)

df_bogie = df_bogie[df_bogie['bogie_drop_pit_lubrication'].notnull()].copy()

flattened_rows = [
    flatten_json(row)
    for row in df_bogie['json_data']
]

bogie_df = pd.DataFrame(flattened_rows)
bogie_df.index = df_bogie.index

# Put identifiers in front
bogie_df.insert(0, 'workorder_id', df_bogie['workorder_id'].astype('Int64'))
bogie_df.insert(1, 'filename', df_bogie['filename'])

front_cols = ['workorder_id', 'filename']

bogie_cols = sorted(
    [c for c in bogie_df.columns if re.match(r'^[a-z]\.', c)]
)

tail_cols = [
    c for c in bogie_df.columns
    if c not in front_cols + bogie_cols
]

bogie_df = bogie_df[front_cols + bogie_cols + tail_cols]

In [21]:
bogie_df

,workorder_id,filename,pinion_shaft_bearings.lubricant,pinion_shaft_bearings.amount,pinion_shaft_bearings.interval,pinion_shaft_bearings.completed,locking_pins.lubricant,locking_pins.amount,locking_pins.interval,locking_pins.completed,...,table_rotating_cylinder.completed,rotating_table_support_wheels_and_pivot_point.lubricant,rotating_table_support_wheels_and_pivot_point.amount,rotating_table_support_wheels_and_pivot_point.interval,rotating_table_support_wheels_and_pivot_point.completed,technician.technician_id,technician.date,supervisor.supervisor_id,supervisor.date,any_additional_work_that_requires_planning
428,4000495745,TN_PM_MTH_BogieDropPit1Lub_4000495745.pdf,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,EP 2 Grease,3 strokes with hand grease gun,Each service,yes,...,yes,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,"9728, 11625",13/11/2022,7091,13/11/2022,-NA-
977,4000693797,TN_PM_MTH_BogieDropPit1Lub_4000693797.pdf,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,EP 2 Grease,3 strokes with hand grease gun,Each service,yes,...,yes,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,"6693, 20856, 21317, 21767",10/08/2025,10180,10/08/2025,N/A
1752,4000479302,TN_PM_MTH_BogieDropPit1Lub_4000479302.pdf,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,EP 2 Grease,3 strokes with hand grease gun,Each service,yes,...,yes,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,"14953, 6693, 11890",NA,7270,NA,N/A
1865,4000459556,TN_PM_MTH_BogieDropPit1Lub_4000459556.pdf,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,EP 2 Grease,3 strokes with hand grease gun,Each service,yes,...,yes,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,"6453, 11626",13/05/2022,9364,13/05/2022,NA
2250,4000473467,TN_PM_MTH_BogieDropPit2Lub_4000473467.pdf,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,EP 2 Grease,3 strokes with hand grease gun,Each service,yes,...,yes,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,"6453,11626",NA,9164,NA,-N/A-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6541,4000490622,TN_PM_MTH_BogieDropPit2Lub_4000490622.pdf,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,EP 2 Grease,3 strokes with hand grease gun,Each service,yes,...,yes,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,"6692, 11590, 14953",14/10/2022,7279,14/10/2022,NA
6542,4000643695,TN_PM_MTH_BogieDropPit2Lub_4000643695.pdf,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,EP 2 Grease,3 strokes with hand grease gun,Each service,yes,...,yes,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,"11626, 19545, 20609, 23018, 23020",06/12/2024,6453,06/12/2024,N/A
6547,4000667955,TN_PM_MTH_BogieDropPit2Lub_4000667955.pdf,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,EP 2 Grease,3 strokes with hand grease gun,Each service,yes,...,yes,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,"20003, 20970, 22155, 23019",08/04/2025,11625,08/04/2025,N/A
6555,4000686334,TN_PM_MTH_BogieDropPit2Lub_4000686334.pdf,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,EP 2 Grease,3 strokes with hand grease gun,Each service,yes,...,yes,EP 2 Grease,5 strokes with hand grease gun,Each service,yes,9456,08/07/2023,"11626, 22548, 23018, 20609",08/07/2023,N/A


### Output Excel

In [23]:
import os
import pandas as pd
from openpyxl import Workbook

output_path = '../../output/tnm/bogie_drop_pit_lub.xlsx'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

if not os.path.exists(output_path):
    Workbook().save(output_path)

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    bogie_df.to_excel(writer, index=False, sheet_name='bogie_drop_pit_lub'),

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")


✅ Exported successfully to '../../output/tnm/bogie_drop_pit_lub.xlsx' (replaced existing sheet)
